# 05 — Batch Inference

**What:** Separate a folder of songs, write target instrument stems to disk.
**Prerequisites:** Trained model.
**Runtime:** Depends on folder size.  **GPU needed:** Yes.

## 1. Batch separate a folder

In [ ]:
from pathlib import Path
import demucs.api
import torch

input_folder = Path('finetune_data/songs/')
output_folder = Path('instrument_stems/')
sig = '<YOUR_SIG>'  # Replace with your trained model signature

if input_folder.exists():
    audio_files = sorted(
        p for p in input_folder.rglob('*')
        if p.suffix.lower() in {'.wav', '.flac', '.mp3', '.ogg', '.m4a'}
    )
    if not audio_files:
        print(f'No audio files found in {input_folder}/')
    else:
        separator = demucs.api.Separator(
            model=sig,
            device='cuda' if torch.cuda.is_available() else 'cpu',
        )
        for fp in audio_files:
            rel = fp.relative_to(input_folder).with_suffix('')
            track_dir = output_folder / sig / rel
            track_dir.mkdir(parents=True, exist_ok=True)
            origin, separated = separator.separate_audio_file(fp)
            for stem_name, stem_wav in separated.items():
                demucs.api.save_audio(
                    stem_wav.cpu(),
                    str(track_dir / f'{stem_name}.wav'),
                    samplerate=separator.samplerate,
                )
            print(f'  {fp.name} → {track_dir}')
        print(f'Done — {len(audio_files)} files processed')
else:
    print(f'Create finetune_data/songs/ with audio files to process')


## 2. Verify output

In [ ]:
instrument_dir = output_folder / sig
if instrument_dir.exists():
    stems = sorted(instrument_dir.rglob('**/chinese-instrument.wav'))
    print(f'{len(stems)} instrument stems extracted:')
    for s in stems[:10]:
        print(f'  {s}')
    if len(stems) > 10:
        print(f'  ... and {len(stems) - 10} more')

## Done!

See [docs/index.md](../docs/index.md) for full documentation.